# Prompt optimisation for NER with DSPy and spaneval

This notebook shows how to automatically optimise a prompt for named entity recognition using
[DSPy](https://dspy.ai/) and [spaneval](https://github.com/jamblejoe/spaneval).

**The core idea:** spaneval's `results.score(goals)` is a single float that encodes
*exactly what good-enough means* for each entity type. It plugs directly into DSPy as
the metric the optimizer chases — no manual F1 averaging or threshold tuning required.

## Setup

Activate the pixi environment before starting:

```bash
pixi run -e dspy jupyter notebook
```

You also need a `.env` file in the repo root:

```
ANTHROPIC_API_KEY=sk-ant-...
```

Prepare the data sample once (downloads from HuggingFace, saves to `data/`):

```bash
pixi run -e dspy python scripts/prepare_conll2003.py
```

## Workflow

1. Load pre-prepared CoNLL-2003 sample from `data/`
2. Define a DSPy NER module (baseline prompt)
3. Evaluate the baseline with spaneval
4. **Define the metric** via spaneval `Goal` objects → `results.score()`
5. Run `MIPROv2` to optimise the prompt
6. Compare baseline vs. optimised

## Imports and configuration

In [2]:
import json
import os
from pathlib import Path
from typing import Literal

import dspy
from dotenv import load_dotenv
from pydantic import BaseModel

from spaneval import Entity, evaluate
from spaneval.results import Goal
from spaneval.strategies import ProportionalCoverage, Strict

load_dotenv()  # reads ANTHROPIC_API_KEY from .env into the environment

# ── Models ────────────────────────────────────────────────────────────────────
# The student does the actual extraction — keep it cheap (many calls during opt).
# The teacher guides MIPROv2 when generating instructions and few-shot examples.
student = dspy.LM("anthropic/claude-haiku-4-5-20251001", max_tokens=512)
teacher = dspy.LM("anthropic/claude-sonnet-4-6",         max_tokens=4096)

dspy.configure(lm=student)

## 1. Data — CoNLL-2003 sample

150 sentences from the Reuters newswire corpus (CoNLL-2003), pre-prepared by
`scripts/prepare_conll2003.py`. Each line in the JSONL files is a plain-text sentence
with character-span entity annotations.

- **Entity types:** PER, ORG, LOC — MISC dropped (too inconsistently defined for LLM prompting)
- **Min length:** 100 characters — filters out score lines, table rows, and headline fragments
- **Split:** 100 sentences for optimizer training, 50 for evaluation

In [3]:
DATA_DIR = Path("..") / "data"


def load_examples(path: Path) -> list[dspy.Example]:
    examples = []
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            entities = [Entity.from_dict(e) for e in row["entities"]]
            examples.append(
                dspy.Example(text=row["text"], gold_entities=entities).with_inputs("text")
            )
    return examples


trainset = load_examples(DATA_DIR / "train.jsonl")
evalset  = load_examples(DATA_DIR / "eval.jsonl")

print(f"Train: {len(trainset)}  |  Eval: {len(evalset)}")
print()
ex = trainset[0]
print("Example text:", ex.text)
print("Gold entities:", ex.gold_entities)

Train: 100  |  Eval: 50

Example text: Iraqi Kurdish areas bordering Iran are under the control of guerrillas of the Iraqi Kurdish Patriotic Union of Kurdistan -LPR- PUK -RPR- group .
Gold entities: [Entity(entity_type='LOC', start=30, end=34, original_text='Iran', replacement_text=None), Entity(entity_type='ORG', start=78, end=120, original_text='Iraqi Kurdish Patriotic Union of Kurdistan', replacement_text=None), Entity(entity_type='ORG', start=127, end=130, original_text='PUK', replacement_text=None)]


## 2. DSPy NER module (baseline)

We define a simple DSPy `Signature` that asks the LLM for a list of
`{text, entity_type}` pairs. The LLM outputs the entity string as it appears in
the text — we then resolve that back to a character span via `str.find()`.

This is a deliberately minimal starting point: a single `Predict` with no
few-shot examples and a generic docstring instruction.

In [4]:
class NamedEntity(BaseModel):
    text: str                                  # exact substring from the input
    entity_type: Literal["PER", "ORG", "LOC"]


class NERSignature(dspy.Signature):
    """Extract named entities from the text.
    Return each entity as its exact text and one of: PER, ORG, LOC."""

    text: str = dspy.InputField()
    entities: list[NamedEntity] = dspy.OutputField(
        desc="Named entities found in the text"
    )


class NERModule(dspy.Module):
    def __init__(self):
        self.predict = dspy.Predict(NERSignature)

    def forward(self, text: str) -> dspy.Prediction:
        return self.predict(text=text)


# ── Helper: LLM mentions → character spans ────────────────────────────────────
def mentions_to_spans(text: str, entities: list[NamedEntity]) -> list[Entity]:
    """Convert NamedEntity objects (text + type) to spaneval Entity objects."""
    result = []
    for ent in (entities or []):
        idx = text.find(ent.text)
        if idx != -1 and ent.text:
            result.append(Entity(
                entity_type=ent.entity_type,
                start=idx,
                end=idx + len(ent.text),
                original_text=ent.text,
            ))
    return result


# Quick smoke test
baseline_ner = NERModule()
pred = baseline_ner(text=evalset[0].text)
print("Input:", evalset[0].text)
print("Predicted entities:", pred.entities)
print("As spans:", mentions_to_spans(evalset[0].text, pred.entities))

Input: He said a proposal last month by EU Farm Commissioner Franz Fischler to ban sheep brains , spleens and spinal cords from the human and animal food chains was a highly specific and precautionary move to protect human health .
Predicted entities: [NamedEntity(text='Franz Fischler', entity_type='PER'), NamedEntity(text='EU', entity_type='ORG')]
As spans: [Entity(entity_type='PER', start=54, end=68, original_text='Franz Fischler', replacement_text=None), Entity(entity_type='ORG', start=33, end=35, original_text='EU', replacement_text=None)]


## 3. Baseline evaluation with spaneval

In [5]:
baseline_ner = NERModule()

true_spans_eval = [ex.gold_entities for ex in evalset]

baseline_pred_spans = [
    mentions_to_spans(ex.text, baseline_ner(text=ex.text).entities)
    for ex in evalset
]

baseline_results = evaluate(true_spans_eval, baseline_pred_spans)
print("=== Baseline — default ± report ===")
baseline_results.report()

=== Baseline — default ± report ===
Strategies: Strict, AnyOverlap

  Entity Type    Number    Missed    Spurious    Precision (%)    Recall (%)
      Overall       107         8          24          78 ±  2       90 ±  2
          LOC        36         5          11          70 ±  4       82 ±  4
          ORG        38         3          10          76 ±  2       89 ±  3
          PER        33         0           3          92 ±  0      100 ±  0


.../spaneval/evaluator.py:57: UserWarning: Overlapping predicted entities resolved: keeping LOC[6:13] (7 chars, longest) over: LOC[6:12] (6 chars). Pass warn_on_overlapping_preds=False to suppress.
  doc_matches.append(matcher.match_entities(true_doc, pred_doc))
.../spaneval/evaluator.py:57: UserWarning: Overlapping predicted entities resolved: keeping PER[6:11] (5 chars, longest) over: PER[6:10] (4 chars). Pass warn_on_overlapping_preds=False to suppress.
  doc_matches.append(matcher.match_entities(true_doc, pred_doc))


In [6]:
# Pin strategies per type and inspect the numbers in detail.
print("=== Baseline — per-type strategy report ===")
baseline_results.report(
    strategy={
        "PER": Strict(),
        "ORG": ProportionalCoverage(),
        "LOC": Strict(),
    }
)

=== Baseline — per-type strategy report ===
  Entity Type              Strategy    Number    Missed    Spurious    Precision (%)    Recall (%)    F1 (%)
      Overall                 mixed       107         8          24               77            89        83
          LOC                Strict        36         5          11               67            78        72
          ORG  ProportionalCoverage        38         3          10               76            90        83
          PER                Strict        33         0           3               92           100        96


## 4. Defining the metric with spaneval

This is the key section. Instead of picking a single F1 number and hoping it
captures "good enough", we express *what we want* per entity type using
`Goal(strategy, recall, precision)`. This is a domain decision, not a statistical one:

| Type | Strategy | Why |
|------|----------|-----|
| PER  | Strict   | Name boundaries must be exact — a partial name is still identifying |
| ORG  | ProportionalCoverage | A partially matched org name is better than nothing |
| LOC  | Strict   | Location precision matters |

`results.score(goals)` returns the **bottleneck score**: the minimum across all
type × metric combinations. The optimizer cannot improve by sacrificing one type
for another — it always has to fix the weakest link.

This single float is all DSPy needs as a metric.

In [7]:
goals = {
    "PER": Goal(strategy=Strict(),               recall=0.75, precision=0.70),
    "ORG": Goal(strategy=ProportionalCoverage(), recall=0.65, precision=0.60),
    "LOC": Goal(strategy=Strict(),               recall=0.75, precision=0.70),
}

In [8]:
print("=== Baseline — goal scorecard ===")
baseline_results.report_goals(goals)
print()
print(f"Bottleneck score: {baseline_results.score(goals):.3f}",
      " (1.0 = all targets exactly met; > 1.0 = all exceeded)")

=== Baseline — goal scorecard ===
  Entity Type              Strategy    Recall    R-Target    R-Score    Precision    P-Target    P-Score
      Overall               (goals)                                                              0.95  ←
          PER                Strict      1.00        0.75       1.33         0.92        0.70       1.31
          ORG  ProportionalCoverage      0.90        0.65       1.39         0.76        0.60       1.27
          LOC                Strict      0.78        0.75       1.04         0.67        0.70     0.95 ←

Bottleneck score: 0.952  (1.0 = all targets exactly met; > 1.0 = all exceeded)


In [9]:
def spaneval_metric(example: dspy.Example, pred: dspy.Prediction, trace=None) -> float:
    pred_spans = mentions_to_spans(example.text, pred.entities or [])
    return evaluate([example.gold_entities], [pred_spans]).score(goals)

## 5. Prompt optimisation with DSPy MIPROv2

`MIPROv2` uses a Bayesian optimisation loop to jointly tune:
- the **instruction** (the docstring of the signature), and
- the **few-shot demonstrations** (bootstrapped from the training set).

The `teacher` model (`claude-sonnet-4-6`) generates candidate instructions
and bootstraps few-shot demonstrations from the training set.
The `student` model (`claude-haiku`) is what actually runs on each training
example during optimization. `spaneval_metric` scores each trial, and MIPROv2's 
Bayesian optimizer uses those scores to decide which instruction and demonstration 
combination to try next.

`auto="light"` runs a small number of trials — enough for a clear improvement
without excessive API cost.

In [10]:
optimizer = dspy.MIPROv2(
    metric=spaneval_metric,
    prompt_model=teacher,
    task_model=student,
    auto="light",
    num_threads=4,
)

optimized_ner = optimizer.compile(
    NERModule(),
    trainset=trainset,
    requires_permission_to_run=False,
)

2026/03/29 12:14:20 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.
2026/03/29 12:14:21 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 80

2026/03/29 12:14:21 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/03/29 12:14:21 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/03/29 12:14:21 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 25%|██▌       | 5/20 [00:00<00:00, 24.00it/s]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Bootstrapping set 4/6


 25%|██▌       | 5/20 [00:00<00:00, 40.76it/s]


...

2026/03/29 12:21:50 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 76.03 / 71 (107.1%):  89%|████████▉ | 71/80 [00:43<00:05,  1.67it/s]

2026/03/29 12:21:52 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 81.03 / 79 (102.6%): 100%|██████████| 80/80 [00:50<00:00,  1.59it/s]

2026/03/29 12:21:58 INFO dspy.evaluate.evaluate: Average Metric: 81.03418803418803 / 80 (101.3%)
2026/03/29 12:21:58 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 101.29
2026/03/29 12:21:58 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [92.92, 94.6, 101.29]
2026/03/29 12:21:58 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 101.29
2026/03/29 12:21:58 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/03/29 12:21:58 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/03/29 12:21:58 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 101.29!


In [11]:
# Inspect the optimised instruction DSPy found
print(optimized_ner.predict.signature.instructions)

Carefully read the provided text and identify all named entities present. For each named entity found, extract its exact surface form as it appears in the original text (preserving capitalization and spacing) and classify it into one of the following three categories:

- **PER** (Person): Any named individual, including full names, last names only, or first names only (e.g., "Sampras", "John Smith")
- **ORG** (Organization): Any named organization, company, institution, political group, media outlet, or collective body (e.g., "Reuters", "Patriotic Union of Kurdistan", "PUK")
- **LOC** (Location): Any named geographical location, including countries, cities, regions, landmarks, or venues (e.g., "Iran", "New York", "Wimbledon", "Iraqi Kurdish")

Important guidelines:
- Include ALL named entities present in the text — do not omit any.
- Preserve the exact text span as it appears in the sentence, including any adjective-noun combinations that form a recognized place name (e.g., "Iraqi Kurd

## 6. Results — baseline vs. optimised

In [12]:
optimized_pred_spans = [
    mentions_to_spans(ex.text, optimized_ner(text=ex.text).entities)
    for ex in evalset
]

optimized_results = evaluate(true_spans_eval, optimized_pred_spans)

.../spaneval/evaluator.py:57: UserWarning: Overlapping predicted entities resolved: keeping LOC[6:13] (7 chars, longest) over: LOC[6:12] (6 chars). Pass warn_on_overlapping_preds=False to suppress.
  doc_matches.append(matcher.match_entities(true_doc, pred_doc))


In [13]:
print("=== BASELINE ===")
baseline_results.report_goals(goals)
print(f"  Bottleneck score: {baseline_results.score(goals):.3f}")

print()
print("=== OPTIMISED ===")
optimized_results.report_goals(goals)
print(f"  Bottleneck score: {optimized_results.score(goals):.3f}")

=== BASELINE ===
  Entity Type              Strategy    Recall    R-Target    R-Score    Precision    P-Target    P-Score
      Overall               (goals)                                                              0.95  ←
          PER                Strict      1.00        0.75       1.33         0.92        0.70       1.31
          ORG  ProportionalCoverage      0.90        0.65       1.39         0.76        0.60       1.27
          LOC                Strict      0.78        0.75       1.04         0.67        0.70     0.95 ←
  Bottleneck score: 0.952

=== OPTIMISED ===
  Entity Type              Strategy    Recall    R-Target    R-Score    Precision    P-Target    P-Score
      Overall               (goals)                                                              1.01  ←
          PER                Strict      0.97        0.75       1.29         0.97        0.70       1.39
          ORG  ProportionalCoverage      0.86        0.65       1.33         0.80        0.60     

In [14]:
# Full ± report for a broader picture
print("=== BASELINE — full report ===")
baseline_results.report()

print()
print("=== OPTIMISED — full report ===")
optimized_results.report()

=== BASELINE — full report ===
Strategies: Strict, AnyOverlap

  Entity Type    Number    Missed    Spurious    Precision (%)    Recall (%)
      Overall       107         8          24          78 ±  2       90 ±  2
          LOC        36         5          11          70 ±  4       82 ±  4
          ORG        38         3          10          76 ±  2       89 ±  3
          PER        33         0           3          92 ±  0      100 ±  0

=== OPTIMISED — full report ===
Strategies: Strict, AnyOverlap

  Entity Type    Number    Missed    Spurious    Precision (%)    Recall (%)
      Overall       107        11          19          82 ±  1       88 ±  1
          LOC        36         5          10          73 ±  2       83 ±  3
          ORG        38         5           8          79 ±  1       86 ±  1
          PER        33         1           1          97 ±  0       97 ±  0
